In [1]:
# # ✅ 1. ENV SETUP
# import os
# from dotenv import load_dotenv

# load_dotenv(".env")

# FIGMA_TOKEN = os.getenv("FIGMA_TOKEN")
# FILE_ID_1 = os.getenv("FIGMA_DOCUMENT_ID1")
# FILE_ID_2 = os.getenv("FIGMA_DOCUMENT_ID2")
# SAS_URL = os.getenv("SAS_URL")

# pages_1 = ["Module 1", "Module 2", "Module 3", "Module 4"]
# pages_2 = ["Module 5", "Module 6", "Module 7", "Module 8"]

# required = ["FIGMA_TOKEN", "FILE_ID_1", "FILE_ID_2", "SAS_URL"]
# missing = [v for v in required if not globals()[v]]
# if missing:
#     raise EnvironmentError(f"❌ Missing env vars: {', '.join(missing)}")

# print("✅ Environment loaded.")

# # ✅ 2. UTILS
# import io
# import time
# from PIL import Image
# from contextlib import contextmanager

# # ---------- UTILS ----------
# def format_blob_name(folder: str, name: str) -> str:
#     return f"{folder}/{name.replace(':', '_').replace(' ', '_')}.webp"

# def format_path(name: str) -> str:
#     return name.replace(" ", "_").strip()

# @contextmanager
# def timed(label):
#     t0 = time.time()
#     yield
#     print(f"⏱️ {label} took {time.time() - t0:.2f}s")

# def convert_image_bytes_to_webp(image_bytes: bytes, min_height=1080, quality=95) -> bytes:
#     with Image.open(io.BytesIO(image_bytes)) as img:
#         img = img.convert("RGB")
#         if img.height < min_height:
#             scale_factor = min_height / img.height
#             new_width = int(img.width * scale_factor)
#             img = img.resize((new_width, min_height), Image.Resampling.LANCZOS)
#         buffer = io.BytesIO()
#         img.save(buffer, format="WEBP", quality=quality)
#         return buffer.getvalue()

# # ✅ 3. AZURE STORAGE
# import ssl
# from azure.storage.blob.aio import ContainerClient
# from azure.storage.blob import ContentSettings
# from azure.core.pipeline.transport import AioHttpTransport

# USE_INSECURE_SSL = False

# def get_container_client():
#     if USE_INSECURE_SSL:
#         ssl_context = ssl.create_default_context()
#         ssl_context.check_hostname = False
#         ssl_context.verify_mode = ssl.CERT_NONE
#         transport = AioHttpTransport(ssl_context=ssl_context)
#         return ContainerClient.from_container_url(SAS_URL, transport=transport)
#     return ContainerClient.from_container_url(SAS_URL)

# async def get_existing_blob_names(prefix: str = "") -> set[str]:
#     blob_names = set()
#     async with get_container_client() as client:
#         async for blob in client.list_blobs(name_starts_with=prefix):
#             blob_names.add(blob.name)
#     print(f"📦 {len(blob_names)} blobs found with prefix '{prefix}'")
#     return blob_names

# async def upload_image_blob(blob_name: str, data: bytes) -> str:
#     async with get_container_client() as client:
#         blob = client.get_blob_client(blob_name)
#         await blob.upload_blob(
#             data=data,
#             overwrite=True,
#             content_settings=ContentSettings(content_type="image/webp"),
#         )
#         return blob.url.split("?")[0]

# # ✅ 4. FIGMA FETCH + REFS
# import requests
# import json

# REFS_PATH = "figma_image_refs.json"

# def fetch_figma_file(file_id: str) -> dict:
#     res = requests.get(
#         f"https://api.figma.com/v1/files/{file_id}",
#         headers={"X-Figma-Token": FIGMA_TOKEN}
#     )
#     res.raise_for_status()
#     return res.json()

# # ---------- FIGMA IMAGE FETCH ----------
# def fetch_image_url_single(file_id: str, node_id: str, format="png", scale=1.0) -> str | None:
#     try:
#         res = requests.get(
#             f"https://api.figma.com/v1/images/{file_id}",
#             headers={"X-Figma-Token": FIGMA_TOKEN},
#             params={"ids": node_id, "format": format, "scale": scale},
#             timeout=30,
#         )
#         res.raise_for_status()
#         return res.json().get("images", {}).get(node_id)
#     except Exception as e:
#         print(f"❌ Error fetching URL for {node_id}: {e}")
#         return None

# def calculate_scale_to_target_height(node: dict, target_height: int = 1080) -> float:
#     """
#     Calculate the Figma image export scale needed to reach the desired height (e.g., 1080px).
#     Caps the scale to Figma’s maximum allowed value of 4.
#     """
#     try:
#         height = node.get("absoluteBoundingBox", {}).get("height", 1)
#         if height <= 0:
#             return 1.0
#         return min(round(target_height / height, 2), 4.0)
#     except Exception:
#         return 1.0

# def map_nodes_by_id(page: dict) -> dict[str, dict]:
#     """
#     Recursively walk a Figma page and return a dictionary mapping node IDs to their node objects.
#     Useful for looking up scale-relevant metadata like height.
#     """
#     node_map = {}

#     def walk(node):
#         if "id" in node:
#             node_map[node["id"]] = node
#         for child in node.get("children", []) or []:
#             walk(child)

#     walk(page)
#     return node_map


# def load_image_refs() -> dict[str, str]:
#     if os.path.exists(REFS_PATH):
#         with open(REFS_PATH, "r") as f:
#             return json.load(f)
#     return {}

# def save_image_refs(refs: dict[str, str]):
#     with open(REFS_PATH, "w") as f:
#         json.dump(refs, f, indent=2)

# # ✅ 5. FETCH IMAGE CONTENT
# import aiohttp

# async def fetch_image_content(url: str) -> bytes | None:
#     try:
#         async with aiohttp.ClientSession() as session:
#             async with session.get(url, timeout=30) as resp:
#                 if resp.status == 200:
#                     return await resp.read()
#                 print(f"⚠️ Failed to fetch {url} (status {resp.status})")
#     except Exception as e:
#         print(f"❌ Error fetching image: {e}")
#     return None

# # ✅ 6. IMAGE UPLOAD MAIN FUNCTION
# async def upload_figma_images(
#     file_id: str,
#     id_to_blobname: dict[str, str],
#     image_refs: dict[str, str],
#     existing_blobs: set[str],
#     cache_path: str,
#     overwrite: bool = False,
# ):
#     previous_refs = {}
#     if os.path.exists(cache_path):
#         try:
#             with open(cache_path, "r") as f:
#                 previous_refs = json.load(f)
#         except Exception as e:
#             print(f"❌ Failed to load cache: {e}")

#     to_upload = {
#         node_id: blob_name
#         for node_id, blob_name in id_to_blobname.items()
#         if overwrite
#         or blob_name not in existing_blobs
#         or image_refs.get(node_id) != previous_refs.get(node_id)
#     }

#     print(f"🧮 {len(id_to_blobname)} total | {len(to_upload)} to upload")

#     uploaded = 0
#     for i, (node_id, blob_name) in enumerate(to_upload.items(), 1):
#         url = fetch_image_url_single(file_id, node_id, scale=1.0)
#         if not url:
#             continue
#         image_bytes = await fetch_image_content(url)
#         if not image_bytes:
#             continue
#         webp = convert_image_bytes_to_webp(image_bytes)
#         await upload_image_blob(blob_name, webp)
#         print(f"✅ [{i}] Uploaded {blob_name}")
#         previous_refs[node_id] = image_refs[node_id]
#         try:
#             with open(cache_path, "w") as f:
#                 json.dump(previous_refs, f, indent=2)
#         except Exception as e:
#             print(f"❌ Failed to update cache after {blob_name}: {e}")
#         uploaded += 1

#     print(f"📊 Done | Uploaded: {uploaded} | Skipped: {len(id_to_blobname) - uploaded}")



# # ✅ 7. SCAN FIGMA PAGE
# import re

# def assign_parents(node: dict, parent: dict = None):
#     node["_parent"] = parent
#     for child in node.get("children", []) or []:
#         assign_parents(child, node)

# def extract_generic_image(node: dict, page_name: str) -> dict | None:
#     """
#     Extract image nodes by returning the RECTANGLE node that contains an IMAGE fill.
#     This ensures the exported image is not cropped by external frames.
#     """
#     if node.get("type") == "RECTANGLE":
#         for fill in node.get("fills", []):
#             if fill.get("type") == "IMAGE" and "imageRef" in fill:
#                 return {
#                     "node_id": node["id"],  # Export RECTANGLE directly
#                     "blob_name": format_blob_name(f"Modules/{format_path(page_name)}", node["id"]),
#                     "image_ref": fill["imageRef"],
#                     "type": "rectangle"
#                 }
#     return None


#     # Fallback: rectangle with image fill — export the rectangle node directly
#     if node.get("type") == "RECTANGLE":
#         fills = node.get("fills", [])
#         for fill in fills:
#             if fill.get("type") == "IMAGE" and "imageRef" in fill:
#                 return {
#                     "node_id": node["id"],  # ✅ Export this RECTANGLE node
#                     "blob_name": format_blob_name(f"Modules/{format_path(page_name)}", node["id"]),
#                     "image_ref": fill["imageRef"],
#                     "type": "rectangle"
#                 }

#     return None


# def scan_figma_page(page, structure_json=None):
#     scanned = {}
#     assign_parents(page)

#     def walk(node):
#         if node.get("type") == "RECTANGLE":
#             for fill in node.get("fills", []):
#                 if fill.get("type") == "IMAGE" and "imageRef" in fill:
#                     scanned[node["id"]] = {
#                         "node_id": node["id"],
#                         "blob_name": format_blob_name(f"Modules/{format_path(page['name'])}", node["id"]),
#                         "image_ref": fill["imageRef"],
#                         "type": "image"
#                     }
#         for child in node.get("children", []) or []:
#             walk(child)

#     walk(page)
#     print(f"🔍 {page['name']}: {len(scanned)} image rectangles found")
#     return scanned




# # ✅ 8. MAIN PIPELINE RUNNER
# import json

# structure_path = "../03_Outputs/SEA_Modules/en/module_structure.json"
# figma_path_1 = "../02_Inputs/figma_jsons/figma_document1.json"
# figma_path_2 = "../02_Inputs/figma_jsons/figma_document2.json"
# cache_path = "../03_Outputs/image_refs_cache.json"

# print("📦 Fetching existing Azure blobs...")
# existing_blobs = await get_existing_blob_names()

# print("🔄 Loading cached image references...")
# updated_refs = load_image_refs()

# with open(structure_path) as f:
#     structure_json = json.load(f)
# with open(figma_path_1) as f:
#     figma_data_1 = json.load(f)
# with open(figma_path_2) as f:
#     figma_data_2 = json.load(f)

# file_page_sets = [
#     (FILE_ID_1, figma_data_1["children"], pages_1),
#     (FILE_ID_2, figma_data_2["children"], pages_2)
# ]

# for file_id, figma_pages, page_names in file_page_sets:
#     for page_name in page_names:
#         page = next((p for p in figma_pages if p["name"] == page_name), None)
#         if not page:
#             print(f"⚠️ Page not found: {page_name}")
#             continue

#         print(f"\n📄 Processing page: {page_name}")
#         scanned = scan_figma_page(page, structure_json)
#         if not scanned:
#             continue

#         id_to_blobname = {
#             node_id: info["blob_name"].replace(" ", "_")
#             for node_id, info in scanned.items()
#         }

#         fetched_refs = {
#             node_id: info["image_ref"]
#             for node_id, info in scanned.items()
#             if "image_ref" in info
#         }

#         await upload_figma_images(
#             file_id=file_id,
#             id_to_blobname=id_to_blobname,
#             image_refs=fetched_refs,
#             existing_blobs=existing_blobs,
#             cache_path=cache_path,
#             overwrite=True,
#         )


# print("✅ Upload pipeline complete.")


In [2]:
# ✅ ENV SETUP
import os
from dotenv import load_dotenv

load_dotenv(".env")

FIGMA_TOKEN = os.getenv("FIGMA_TOKEN")
FILE_ID_1 = os.getenv("FIGMA_DOCUMENT_ID1")
FILE_ID_2 = os.getenv("FIGMA_DOCUMENT_ID2")
SAS_URL = os.getenv("SAS_URL")

pages_1 = ["Module 1", "Module 2", "Module 3", "Module 4","Module 9"]
pages_2 = ["Module 5", "Module 6", "Module 7", "Module 8"]

required = ["FIGMA_TOKEN", "FILE_ID_1", "FILE_ID_2", "SAS_URL"]
missing = [v for v in required if not globals()[v]]
if missing:
    raise EnvironmentError(f"❌ Missing env vars: {', '.join(missing)}")

print("✅ Environment loaded.")

# ✅ UTILS
import io
import time
from PIL import Image
from contextlib import contextmanager

def format_blob_name(folder: str, name: str) -> str:
    return f"{folder}/{name.replace(':', '_').replace(' ', '_')}.webp"

def format_path(name: str) -> str:
    return name.replace(" ", "_").strip()

@contextmanager
def timed(label):
    t0 = time.time()
    yield
    print(f"⏱️ {label} took {time.time() - t0:.2f}s")

def convert_image_bytes_to_webp(image_bytes: bytes, max_height: int = 1080, quality: int = 95) -> bytes:
    with Image.open(io.BytesIO(image_bytes)) as img:
        img = img.convert("RGB")

        # Calculate scale factor if height is above max_height
        if img.height > max_height:
            scale = max_height / img.height
            new_width = int(img.width * scale)
            new_height = max_height
            img = img.resize((new_width, new_height), Image.Resampling.LANCZOS)

        buffer = io.BytesIO()
        img.save(buffer, format="WEBP", quality=quality)
        return buffer.getvalue()


# ✅ AZURE STORAGE
import ssl
from azure.storage.blob.aio import ContainerClient
from azure.storage.blob import ContentSettings
from azure.core.pipeline.transport import AioHttpTransport

USE_INSECURE_SSL = False

def get_container_client():
    if USE_INSECURE_SSL:
        ssl_context = ssl.create_default_context()
        ssl_context.check_hostname = False
        ssl_context.verify_mode = ssl.CERT_NONE
        transport = AioHttpTransport(ssl_context=ssl_context)
        return ContainerClient.from_container_url(SAS_URL, transport=transport)
    return ContainerClient.from_container_url(SAS_URL)

async def get_existing_blob_names(prefix: str = "") -> set[str]:
    blob_names = set()
    async with get_container_client() as client:
        async for blob in client.list_blobs(name_starts_with=prefix):
            blob_names.add(blob.name)
    print(f"📦 {len(blob_names)} blobs found with prefix '{prefix}'")
    return blob_names

async def upload_image_blob(blob_name: str, data: bytes) -> str:
    async with get_container_client() as client:
        blob = client.get_blob_client(blob_name)
        await blob.upload_blob(
            data=data,
            overwrite=True,
            content_settings=ContentSettings(content_type="image/webp"),
        )
        return blob.url.split("?")[0]

def assign_parents(node: dict, parent: dict | None = None):
    node["_parent"] = parent
    for child in node.get("children", []) or []:
        assign_parents(child, node)


# ✅ FETCH IMAGE URLS
import requests
import json

REFS_PATH = "figma_image_refs.json"

def calculate_optimal_scale(canvas_height: float, target_height: int = 1080, max_scale: float = 4.0) -> float:
    """
    Calculate the optimal scale factor for Figma image export.

    Parameters:
    - canvas_height (float): The height of the image on the canvas.
    - target_height (int): The desired minimum height in pixels after scaling.
    - max_scale (float): The maximum scale Figma API allows.

    Returns:
    - float: A scale factor (rounded to 2 decimals) to use in the image export URL.
    """
    if not canvas_height or canvas_height <= 0:
        return 2.0  # sensible default if height is missing

    scale = target_height / canvas_height
    adjusted_scale=round(min(max_scale, max(1.0, scale)), 2)
    return adjusted_scale


def fetch_image_url_single(file_id: str, node_id: str, canvas_height: float, format="png") -> str | None:
    try:
        scale = calculate_optimal_scale(canvas_height)
        res = requests.get(
            f"https://api.figma.com/v1/images/{file_id}",
            headers={"X-Figma-Token": FIGMA_TOKEN},
            params={"ids": node_id, "format": format, "scale": scale},
            timeout=30,
        )
        res.raise_for_status()
        return res.json().get("images", {}).get(node_id)
    except Exception as e:
        print(f"❌ Error fetching URL for {node_id}: {e}")
        return None


def load_image_refs() -> dict[str, str]:
    if os.path.exists(REFS_PATH):
        with open(REFS_PATH, "r") as f:
            return json.load(f)
    return {}

def save_image_refs(refs: dict[str, str]):
    with open(REFS_PATH, "w") as f:
        json.dump(refs, f, indent=2)

# ✅ FETCH IMAGE CONTENT
import aiohttp

async def fetch_image_content(url: str) -> bytes | None:
    try:
        async with aiohttp.ClientSession() as session:
            async with session.get(url, timeout=30) as resp:
                if resp.status == 200:
                    return await resp.read()
                print(f"⚠️ Failed to fetch {url} (status {resp.status})")
    except Exception as e:
        print(f"❌ Error fetching image: {e}")
    return None

# ✅ SCAN FIGMA PAGE

from math import ceil

def extract_generic_image(node: dict, page_name: str) -> dict | None:
    if node.get("type") != "RECTANGLE":
        return None
    for fill in node.get("fills", []):
        if fill.get("type") == "IMAGE" and "imageRef" in fill:
            canvas_height = node.get("absoluteBoundingBox", {}).get("height", 0)
            return {
                "node_id": node["id"],
                "blob_name": format_blob_name(f"Modules/{format_path(page_name)}", node["id"]),
                "image_ref": fill["imageRef"],
                "canvas_height": node.get("absoluteBoundingBox", {}).get("height", 0),
                "type": "generic"
            }
    return None


def scan_figma_page(page):
    scanned = {}
    assign_parents(page)
    type_counts = {"generic": 0}

    def walk(node):
        if node.get("type") == "RECTANGLE":
            result = extract_generic_image(node, page["name"])
            if result:
                scanned[result["node_id"]] = result
                type_counts["generic"] += 1
        for child in node.get("children", []) or []:
            walk(child)

    walk(page)
    total = sum(type_counts.values())
    print(f"🔍 {page['name']}: {total} image rectangles found")
    return scanned


# ✅ UPLOAD
async def upload_figma_images(file_id, id_to_blobname, image_refs, existing_blobs, cache_path, overwrite=False):
    previous_refs = {}
    if os.path.exists(cache_path):
        try:
            with open(cache_path, "r") as f:
                previous_refs = json.load(f)
        except Exception as e:
            print(f"❌ Failed to load cache: {e}")

    to_upload = {
        node_id: blob_name
        for node_id, blob_name in id_to_blobname.items()
        if overwrite or blob_name not in existing_blobs or image_refs.get(node_id) != previous_refs.get(node_id)
    }

    print(f"🧮 {len(id_to_blobname)} total | {len(to_upload)} to upload")

    uploaded = 0
    for i, (node_id, blob_name) in enumerate(to_upload.items(), 1):
        canvas_height = scanned[node_id].get("canvas_height", 0)
        url = fetch_image_url_single(file_id, node_id, canvas_height)
        if not url:
            continue
        image_bytes = await fetch_image_content(url)
        if not image_bytes:
            continue
        webp = convert_image_bytes_to_webp(image_bytes)
        await upload_image_blob(blob_name, webp)
        print(f"✅ [{i}] Uploaded {blob_name}")
        previous_refs[node_id] = image_refs[node_id]
        try:
            with open(cache_path, "w") as f:
                json.dump(previous_refs, f, indent=2)
        except Exception as e:
            print(f"❌ Failed to update cache after {blob_name}: {e}")
        uploaded += 1

    print(f"📊 Done | Uploaded: {uploaded} | Skipped: {len(id_to_blobname) - uploaded}")


✅ Environment loaded.


In [3]:
# ✅ 8. MAIN PIPELINE RUNNER
import json

structure_path = "../03_Outputs/SEA_Modules/en/module_structure.json"
figma_path_1 = "../02_Inputs/figma_jsons/figma_document1.json"
figma_path_2 = "../02_Inputs/figma_jsons/figma_document2.json"
cache_path = "../03_Outputs/image_refs_cache.json"

print("📦 Fetching existing Azure blobs...")
existing_blobs = await get_existing_blob_names()

print("🔄 Loading cached image references...")
updated_refs = load_image_refs()

with open(structure_path) as f:
    structure_json = json.load(f)
with open(figma_path_1) as f:
    figma_data_1 = json.load(f)
with open(figma_path_2) as f:
    figma_data_2 = json.load(f)

file_page_sets = [
    (FILE_ID_1, figma_data_1["children"], pages_1),
    (FILE_ID_2, figma_data_2["children"], pages_2)
]

for file_id, figma_pages, page_names in file_page_sets:
    for page_name in page_names:
        page = next((p for p in figma_pages if p["name"] == page_name), None)
        if not page:
            print(f"⚠️ Page not found: {page_name}")
            continue

        print(f"\n📄 Processing page: {page_name}")
        scanned = scan_figma_page(page)
        if not scanned:
            continue

        id_to_blobname = {
            node_id: info["blob_name"].replace(" ", "_")
            for node_id, info in scanned.items()
        }

        fetched_refs = {
            node_id: info["image_ref"]
            for node_id, info in scanned.items()
            if "image_ref" in info
        }

        await upload_figma_images(
            file_id=file_id,
            id_to_blobname=id_to_blobname,
            image_refs=fetched_refs,
            existing_blobs=existing_blobs,
            cache_path=cache_path,
            overwrite=False,
        )

print("✅ Upload pipeline complete.")

📦 Fetching existing Azure blobs...
📦 5489 blobs found with prefix ''
🔄 Loading cached image references...

📄 Processing page: Module 1
🔍 Module 1: 188 image rectangles found
🧮 188 total | 0 to upload
📊 Done | Uploaded: 0 | Skipped: 188

📄 Processing page: Module 2
🔍 Module 2: 252 image rectangles found
🧮 252 total | 0 to upload
📊 Done | Uploaded: 0 | Skipped: 252

📄 Processing page: Module 3
🔍 Module 3: 175 image rectangles found
🧮 175 total | 52 to upload
✅ [1] Uploaded Modules/Module_3/725_298.webp
✅ [2] Uploaded Modules/Module_3/459_157.webp
✅ [3] Uploaded Modules/Module_3/459_163.webp
✅ [4] Uploaded Modules/Module_3/459_169.webp
✅ [5] Uploaded Modules/Module_3/725_310.webp
✅ [6] Uploaded Modules/Module_3/4282_225.webp
✅ [7] Uploaded Modules/Module_3/429_193.webp
✅ [8] Uploaded Modules/Module_3/429_199.webp
✅ [9] Uploaded Modules/Module_3/429_205.webp
✅ [10] Uploaded Modules/Module_3/418_225.webp
✅ [11] Uploaded Modules/Module_3/418_231.webp
✅ [12] Uploaded Modules/Module_3/723_72.w

In [4]:
# ###only to run if something fails


# # Figma 403 debugger: run this in ONE cell
# # It will tell you whether the problem is: missing/empty token, wrong file key, no access to file,
# # org/policy restriction on /images endpoint, or bad node IDs.

# import os, re, json, requests
# from urllib.parse import urlparse

# from dotenv import load_dotenv
# load_dotenv(".env")  # or load_dotenv("/mnt/data/.env") if needed


# def _mask(s: str, keep_start=4, keep_end=4) -> str:
#     if not s:
#         return "None"
#     if len(s) <= keep_start + keep_end:
#         return s
#     return f"{s[:keep_start]}…{s[-keep_end:]}"

# def _extract_file_key(s: str | None) -> str | None:
#     if not s:
#         return None
#     s = s.strip()
#     # If it's a full Figma URL like https://www.figma.com/file/<KEY>/...
#     m = re.search(r"figma\.com/(file|design)/([A-Za-z0-9]+)", s)
#     if m:
#         return m.group(2)
#     # If it's already a key
#     return s

# def _req(label: str, url: str, headers: dict, params: dict | None = None, timeout=30):
#     try:
#         r = requests.get(url, headers=headers, params=params, timeout=timeout)
#         print(f"\n=== {label} ===")
#         print("URL:", r.url)
#         print("Status:", r.status_code)
#         ct = r.headers.get("content-type", "")
#         print("Content-Type:", ct)
#         body = r.text[:800] if isinstance(r.text, str) else str(r.content)[:800]
#         print("Body (first 800 chars):")
#         print(body)
#         return r
#     except Exception as e:
#         print(f"\n=== {label} FAILED ===")
#         print("URL:", url)
#         print("Error:", repr(e))
#         return None

# # --------- CONFIG: try to auto-detect from your notebook variables/env ----------
# # If you already have FIGMA_TOKEN / FILE_ID_1 in your notebook globals, this will use them.
# FIGMA_TOKEN = globals().get("FIGMA_TOKEN") or os.getenv("FIGMA_TOKEN")
# FILE_ID = globals().get("FILE_ID_1") or globals().get("FILE_ID") or os.getenv("FIGMA_DOCUMENT_ID1") or os.getenv("FILE_ID_1")

# # If your earlier logs show the file key directly (e.g., G63dbqCKnazZenINIwtfY1), you can hardcode here:
# # FILE_ID = "G63dbqCKnazZenINIwtfY1"

# FILE_ID = _extract_file_key(FILE_ID)

# print("Token present?:", bool(FIGMA_TOKEN))
# print("Token length:", len(FIGMA_TOKEN or ""))
# print("Token (masked):", _mask(FIGMA_TOKEN or ""))
# print("File key:", FILE_ID)

# if not FIGMA_TOKEN:
#     raise RuntimeError("FIGMA_TOKEN is missing/empty. Load env vars or set FIGMA_TOKEN before running.")
# if not FILE_ID:
#     raise RuntimeError("FILE_ID is missing/empty. Set FILE_ID to your Figma file key (e.g., G63dbqCKnazZenINIwtfY1).")

# headers = {"X-Figma-Token": FIGMA_TOKEN}

# # 1) Who am I? (validates token)
# _req("Auth check: GET /v1/me", "https://api.figma.com/v1/me", headers)

# # 2) Can I access the file metadata? (validates file access)
# file_resp = _req("File access check: GET /v1/files/<file_key>", f"https://api.figma.com/v1/files/{FILE_ID}", headers)

# # If file access failed, stop here (wrong file key or no access)
# if not file_resp or file_resp.status_code != 200:
#     print("\n❗ Diagnosis:")
#     if file_resp and file_resp.status_code in (401, 403):
#         print("- Your token is valid enough to call the API but does NOT have access to this file (or org policy blocks access).")
#     elif file_resp and file_resp.status_code == 404:
#         print("- The file key is wrong OR the file is not accessible to your token (sometimes shown as 404).")
#     else:
#         print("- Unexpected error accessing the file; check the response above.")
#     raise SystemExit

# # 3) Pull a few candidate node IDs from the file JSON, then test the images endpoint on them
# file_json = file_resp.json()
# document = file_json.get("document", {})
# node_ids = []

# def walk(n):
#     if isinstance(n, dict):
#         nid = n.get("id")
#         if nid and ":" in nid:
#             node_ids.append(nid)
#         for c in n.get("children", []) or []:
#             walk(c)

# walk(document)

# # Prefer some IDs that likely exist (first few unique)
# node_ids = list(dict.fromkeys(node_ids))  # de-dupe preserving order
# sample_ids = node_ids[:5]

# print("\nSample node IDs from file:", sample_ids)

# # 4) Test images endpoint with these node IDs
# for nid in sample_ids:
#     img_resp = _req(
#         f"Images endpoint test: GET /v1/images/<file_key>?ids={nid}",
#         f"https://api.figma.com/v1/images/{FILE_ID}",
#         headers,
#         params={"ids": nid, "format": "png", "scale": 1},
#     )

#     if img_resp is None:
#         continue

#     if img_resp.status_code == 200:
#         payload = img_resp.json()
#         url = (payload.get("images") or {}).get(nid)
#         print("Image URL returned?:", bool(url))
#         if url:
#             # 5) Try fetching the returned CDN URL (no token required). This checks if image rendering is actually happening.
#             cdn_resp = _req("Fetch rendered CDN URL", url, headers={})
#         print("\n✅ Diagnosis: Your token can use /images for this file. The earlier 403s are likely due to:")
#         print("- Using a DIFFERENT file_id than the one you think, OR")
#         print("- Passing node IDs that don't belong to this file, OR")
#         print("- Some other code path not sending the header/token.")
#         break

#     elif img_resp.status_code == 403:
#         print("\n❗ Diagnosis: You can access /files but NOT /images (403). This usually means:")
#         print("- Org policy restriction on image export via API, OR")
#         print("- Your token/account has view-only access where exports are restricted, OR")
#         print("- You are using a token from an account that can open the file but is blocked from image renders.")
#         break

#     elif img_resp.status_code in (401,):
#         print("\n❗ Diagnosis: Token invalid/expired (401). Regenerate token.")
#         break

#     elif img_resp.status_code == 404:
#         print("\n❗ Diagnosis: Wrong file key for /images OR unexpected access masking. Re-check FILE_ID.")
#         break

# # 6) Extra: If you have specific failing node IDs from logs, test them directly:
# failing_ids = ["609:345", "488:1680", "488:1685", "488:1690"]
# print("\nTesting your failing IDs directly:", failing_ids)
# for nid in failing_ids:
#     _req(
#         f"Failing ID test: {nid}",
#         f"https://api.figma.com/v1/images/{FILE_ID}",
#         headers,
#         params={"ids": nid, "format": "png", "scale": 1},
#     )

# print("\nDone. The outputs above should clearly indicate which layer is failing (token, file access, images endpoint, or node IDs).")
